# CMS Data Generation

This notebook generates clean, structured data for the CMS backend, following the exact data model requirements. We'll create a complete example for the "Climate Mitigation Potential" category, including:

- **Categories**: Climate mitigation potential category
- **Indicators**: Wetlands mitigation potential indicator
- **Indicator Data**: Country-level mitigation data
- **Layers**: Visualization layer for the indicator
- **Locations**: Administrative regions (countries)

The generated data will be exported as JSON files ready for import into the CMS.

## Table of Contents

1. [Setup](#setup)
   - [Library Import](#library-import)
   - [Configuration and Data Paths](#configuration-and-data-paths)
   - [Define CMS Data Model Structure](#define-cms-data-model-structure)
   - [Utility Functions](#utility-functions)

2. [Load and Prepare Source Data](#load-and-prepare-source-data)
   - [Country Data](#country-data)
   - [Mitigation Data](#mitigation-data)

3. [Prepare CMS Data](#prepare-cms-data)
   - 3.1 [Create Locations Data](#1-create-locations-data)
   - 3.2 [Create Category Data](#2-create-category-data)
   - 3.3 [Create Indicator Data](#3-create-indicator-data)
   - 3.4 [Create Indicator Data Records](#4-create-indicator-data-records)
   - 3.5 [Create Layer Data](#5-create-layer-data)

4. [Export Data to JSON Files](#export-data-to-json-files)

5. [Validation and Summary](#validation-and-summary)

<a id='setup'></a>
## 1. Setup

<a id='library-import'></a>
### Library Import

In [1]:
import json
from pathlib import Path
from typing import Dict, List

import geopandas as gpd
import pandas as pd
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

# Initialize rich console
console = Console()

<a id='configuration-and-data-paths'></a>
### Configuration and Data Paths

In [3]:
# Data paths
DATA_PATH = Path("../data/processed")
CMS_OUTPUT_PATH = DATA_PATH / "cms_data"

# Create output directory if it doesn't exist
CMS_OUTPUT_PATH.mkdir(exist_ok=True)

# Configuration
CATEGORY_ID = "climate-mitigation-potential"
INDICATOR_ID = "wetlands-mitigation-potential"
LAYER_ID = "wetlands-layer"

console.print(f"Data path: {DATA_PATH}")
console.print(f"CMS output path: {CMS_OUTPUT_PATH}")
console.print(f"Output directory exists: {CMS_OUTPUT_PATH.exists()}")

Data path: ../data/processed

CMS output path: ../data/processed/cms_data

Output directory exists: True

✅ Configuration completed!

<a id='define-cms-data-model-structure'></a>
### Define CMS Data Model Structure

Here we define the exact structure that each CMS entity requires, based on the [TypeScript definitions](https://github.com/Vizzuality/wetlands/tree/staging/app/src/cms/collections).

In [4]:
# Category structure
CATEGORY_TEMPLATE = {
    "id": "",  # slug_id: generated from name
    "name": "",  # localized text, required
    "description": "",  # localized textarea
    "cover": None,  # upload relationship to media
    "defaultIndicators": [],  # relationship to indicators (array)
}

# Indicator structure
INDICATOR_TEMPLATE = {
    "id": "",  # slug_id: generated
    "name": "",  # localized text, required
    "description": "",  # localized richText
    "category": "",  # relationship to category (single)
}

# Indicator Data structure
INDICATOR_DATA_TEMPLATE = {
    "id": "",  # unique text
    "indicator": "",  # relationship to indicator (single)
    "location": "",  # relationship to location (single)
    "data": {},  # json object with actual data
}

# Layer structure
LAYER_TEMPLATE = {
    "id": "",  # slug_id: generated
    "name": "",  # localized text, required
    "renderingConfig": {},  # rendering configuration
    "paramsConfig": {},  # parameters configuration
    "legendConfig": {},  # legend configuration
    "indicator": "",  # relationship to indicator (single, conditional)
    "type": "INDICATOR",  # select: INDICATOR | CONTEXTUAL
}

# Location structure
LOCATION_TEMPLATE = {
    "id": "",  # compound slug from type + code
    "name": "",  # localized text, required
    "code": "",  # unique identifying code
    "geometry": {},  # json geometry (hidden)
    "bbox": [],  # bounding box array [minLng, minLat, maxLng, maxLat]
    "type": "ADMIN_REGION",  # select: ADMIN_REGION | HYDRO_BASIN | GLOBAL | WDPA
    "parent": None,  # relationship to parent location
}

console.print("[green]✅ Data model templates defined![/green]")

✅ Data model templates defined!

<a id='utility-functions'></a>
### Utility Functions

In [ ]:
def generate_slug_id(text: str) -> str:
    """Generate a slug ID from text"""
    return text.lower().replace(" ", "-").replace("_", "-")


def save_json(data: List[Dict], filename: str, output_path: Path = CMS_OUTPUT_PATH) -> None:
    """Save data as JSON file"""
    filepath = output_path / f"{filename}.json"
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    console.print(f"[green]✅ Saved {len(data)} records to {filepath}[/green]")


console.print("\n[green]✅ Configuration completed![/green]")

<a id='load-and-prepare-source-data'></a>
## 2. Load and Prepare Source Data

Load the processed data that we'll transform for the CMS.

<a id='country-data'></a>
### Country Data

In [54]:
try:
    gdf_countries = gpd.read_file(DATA_PATH / "countries_sahel.geojson")
    console.print(f"[green]✅ Loaded {len(gdf_countries)} countries[/green]")
except FileNotFoundError:
    console.print(
        "[red]❌ Countries file not found. Please run the data processing notebook first.[/red]"
    )
    gdf_countries = None

✅ Loaded 22 countries

<a id='mitigation-data'></a>
### Mitigation Data

In [ ]:
# Load mitigation data from GCS
try:
    df_mitigation = pd.read_csv(
        "https://storage.googleapis.com/wetlands-gap-map/original_data/Mitigation_Country_clean.csv"
    )

    # Select and clean columns
    selected_cols = [
        "ISO",
        "Reduce deforestation AVG",
        "Reforestation AVG",
        "Forest management AVG",
        "Grassland and savanna fire mgmt ",
        "Reduce peatland degradation",
        "Peatland restoration",
        "Reduce Mangrove conversion",
        "Mangrove restoration",
    ]

    df_mitigation = df_mitigation[selected_cols].copy()
    df_mitigation.columns = df_mitigation.columns.str.replace(" ", "_").str.lower().str.strip()

    # Filter to Sahel countries if countries data is available
    if gdf_countries is not None:
        initial_count = len(df_mitigation)
        df_mitigation = df_mitigation[df_mitigation["iso"].isin(gdf_countries["ISO3"])]
        console.print(
            f"[green]✅ Loaded mitigation data: {initial_count} → \
                {len(df_mitigation)} countries (filtered to Sahel)[/green]"
        )
    else:
        console.print(
            f"[yellow]✅ Loaded mitigation data: {len(df_mitigation)} \
                countries (no filtering applied)[/yellow]"
        )

except Exception as e:
    console.print(f"[red]❌ Error loading mitigation data: {e}[/red]")
    df_mitigation = None

# Display sample data
console.print("Sample mitigation data:")
df_mitigation.head(3)

✅ Loaded mitigation data: 250 → 22 countries (filtered to Sahel)

Sample mitigation data:

,iso,reduce_deforestation_avg,reforestation_avg,forest_management_avg,grassland_and_savanna_fire_mgmt_,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration
22,BEN,262.9976,204.0629,3.5320,52.9058,2565.0000,3180.0000,847.0226,704.0
35,BFA,115.6229,199.2904,31.2582,16.8085,NaN,870.0000,NaN,NaN
39,CMR,374.3571,218.1430,0.6945,20.0409,1045.2857,1978.6957,1470.9252,704.0


<a id='prepare-cms-data'></a>
## 3. Prepare CMS Data

<a id='1-create-locations-data'></a>
### 3.1. Create Locations Data

**Transform country boundaries into CMS location format.**

In [31]:
def create_locations_data(gdf_countries):
    """Create locations data from countries GeoDataFrame"""
    if gdf_countries is None:
        return []

    locations = []

    for _, row in gdf_countries.iterrows():
        # Create location ID as compound slug: type_code
        location_id = f"adminregion_{row['ISO3'].lower()}"

        location = {
            "id": location_id,
            "name": row["name"],
            "code": row["ISO3"],
            "geometry": row.geometry.__geo_interface__,
            "bbox": row.get("bbox", []),
            "type": "ADMIN_REGION",
            "parent": None,
        }

        locations.append(location)

    return locations


# Create locations data
locations_data = create_locations_data(gdf_countries)

console.print(f"[green]✅ Created {len(locations_data)} location records[/green]")

# Print sample location data
if locations_data:
    console.print("Sample location:")
    sample_location = locations_data[0].copy()
    # Don't print full geometry for readability
    sample_location["geometry"] = f"<Geometry: {sample_location['geometry']['type']}>"
    console.print_json(data=sample_location)

✅ Created 22 location records

Sample location:

{
  "id": "adminregion_ben",
  "name": "Benin",
  "code": "BEN",
  "geometry": "<Geometry: Polygon>",
  "bbox": "{ \"bbox\": [ 0.776667, 6.0398696000000003, 3.8451453999999998, 12.409202799999999 ] }",
  "type": "ADMIN_REGION",
  "parent": null
}

<a id='2-create-category-data'></a>
### 3.2. Create Category Data

**Create the "Climate Mitigation Potential" category.**

In [ ]:
def create_category_data():
    """Create the Climate Mitigation Potential category"""

    category = {
        "id": CATEGORY_ID,
        "name": "Climate Mitigation Potential",
        "description": "Explore the potential of wetlands and related ecosystems for \
            climate change mitigation through various intervention strategies including \
                forest management, peatland restoration, and mangrove conservation.",
        "cover": None,  # To be set later when media is uploaded
        "defaultIndicators": [INDICATOR_ID],  # Reference to our mitigation indicator
    }

    return [category]  # Return as list for consistency


# Create category data
categories_data = create_category_data()

console.print(f"[green]✅ Created {len(categories_data)} category record[/green]")
console.print("Category data:")
console.print_json(data=categories_data[0])

✅ Created 1 category record

Category data:

{
  "id": "climate-mitigation-potential",
  "name": "Climate Mitigation Potential",
  "description": "Explore the potential of wetlands and related ecosystems for climate change mitigation through various intervention strategies including forest management, peatland restoration, and mangrove conservation.",
  "cover": null,
  "defaultIndicators": [
    "wetlands-mitigation-potential"
  ]
}

<a id='3-create-indicator-data'></a>
### 3.3. Create Indicator Data


**Create the wetlands mitigation potential indicator**

In [ ]:
def create_indicator_data():
    """Create the wetlands mitigation potential indicator"""

    indicator = {
        "id": INDICATOR_ID,
        "name": "Wetlands Mitigation Potential",
        "description": "This indicator shows the potential for various climate mitigation \
            interventions across different ecosystem types.",
        "category": CATEGORY_ID,
    }

    return [indicator]  # Return as list for consistency


# Create indicator data
indicators_data = create_indicator_data()

console.print(f"[green]✅ Created {len(indicators_data)} indicator record[/green]")
console.print("Indicator data:")
console.print_json(data=indicators_data[0])

✅ Created 1 indicator record

Indicator data:

{
  "id": "wetlands-mitigation-potential",
  "name": "Wetlands Mitigation Potential",
  "description": "This indicator shows the potential for various climate mitigation interventions across different ecosystem types. Values represent average mitigation potential in CO₂ equivalent per hectare per year. Interventions include: deforestation reduction ({reduce_deforestation_avg}), reforestation ({reforestation_avg}), forest management ({forest_management_avg}), grassland fire management ({grassland_and_savanna_fire_mgmt_}), peatland degradation reduction ({reduce_peatland_degradation}), peatland restoration ({peatland_restoration}), mangrove conversion reduction ({reduce_mangrove_conversion}), and mangrove restoration ({mangrove_restoration}).",
  "category": "climate-mitigation-potential"
}

<a id='4-create-indicator-data-records'></a>
### 3.4. Create Indicator Data Records

Transform the mitigation data into indicator data records that link indicators to locations.

In [35]:
def create_indicator_data_records(df_mitigation):
    """Create indicator data records from mitigation DataFrame"""
    if df_mitigation is None:
        return []

    indicator_data_records = []

    for idx, row in df_mitigation.iterrows():
        # Create location ID to match locations data
        location_id = f"adminregion_{row['iso'].lower()}"

        # Prepare data object with all mitigation values
        data_values = {}
        for col in df_mitigation.columns:
            if col != "iso" and pd.notna(row[col]):
                # Clean column name for JSON
                clean_key = col.replace("_", " ").title()
                data_values[clean_key] = (
                    float(row[col]) if isinstance(row[col], (int, float)) else row[col]
                )

        # Create indicator data record
        record = {
            "id": f"mitigation-data-{idx}",
            "indicator": INDICATOR_ID,
            "location": location_id,
            "data": data_values,
        }

        indicator_data_records.append(record)

    return indicator_data_records


# Create indicator data records
indicator_data_records = create_indicator_data_records(df_mitigation)

console.print(f"[green]✅ Created {len(indicator_data_records)} indicator data records[/green]")

if indicator_data_records:
    console.print("Sample indicator data record:")
    console.print_json(data=indicator_data_records[0])

✅ Created 22 indicator data records

Sample indicator data record:

{
  "id": "mitigation-data-22",
  "indicator": "wetlands-mitigation-potential",
  "location": "adminregion_ben",
  "data": {
    "Reduce Deforestation Avg": 262.9976,
    "Reforestation Avg": 204.0629,
    "Forest Management Avg": 3.532,
    "Grassland And Savanna Fire Mgmt ": 52.9058,
    "Reduce Peatland Degradation": 2565.0,
    "Peatland Restoration": 3180.0,
    "Reduce Mangrove Conversion": 847.0226,
    "Mangrove Restoration": 704.0
  }
}

<a id='5-create-layer-data'></a>
### 3.5. Create Layer Data

**Create the visualization layer for the mitigation indicator**

In [ ]:
def create_wetlands_layer_data():
    """Create the visualization layer for wetlands by type"""

    # Wetland type categories with raster values and colors
    wetland_categories = {
        0: {"color": "#E5CB71", "label": "Non Wetland"},
        11: {"color": "#FF00F3", "label": "F1.1 Permanent upland streams"},
        12: {"color": "#000CFF", "label": "F1.2 Permanent lowland rivers"},
        14: {"color": "#CCDE57", "label": "F1.4 Seasonal upland streams"},
        15: {"color": "#3898DF", "label": "F1.5 Seasonal lowland rivers"},
        16: {"color": "#D8255B", "label": "F1.6 Episodic arid rivers"},
        21: {"color": "#FB00FF", "label": "F2.1 Large permanent freshwater lakes"},
        23: {"color": "#8A00FF", "label": "F2.3 Seasonal freshwater lakes"},
        25: {"color": "#FF1E00", "label": "F2.5 Ephemeral freshwater lakes"},
        26: {"color": "#E2FDFF", "label": "F2.6 Permanent salt and soda lakes"},
        27: {"color": "#9CF7F8", "label": "F2.7 Ephemeral salt lakes"},
        31: {"color": "#5DEFAB", "label": "F3.1 Large reservoirs"},
        35: {"color": "#D4E613", "label": "F3.5 Canals ditches and drains"},
        43: {"color": "#155700", "label": "TF1.3 Permanent marshes"},
        44: {"color": "#5B9A68", "label": "TF1.4 Seasonal floodplain marshes"},
        45: {"color": "#408F4B", "label": "TF1.5 Episodic arid floodplains"},
    }

    # Rendering configuration for categorical raster display
    rendering_config = {
        "type": "raster",
        "colormap": {
            "type": "categorical",
            "values": list(wetland_categories.keys()),
            "colors": [cat["color"] for cat in wetland_categories.values()],
        },
        "opacity": 0.8,
    }

    # Parameters configuration for raster data
    params_config = {
        "dataSource": "raster",
        "rasterUrl": "https://storage.googleapis.com/wetlands-gap-map/cogs/\
            IUCN_Classified_Sahel_2019-2023.tif",
        "attribution": "Wetlands classification data",
    }

    # Categorical legend configuration
    legend_config = {
        "type": "categorical",
        "title": "Wetland Types",
        "position": "bottom-left",
        "orientation": "vertical",
        "categories": [
            {"value": value, "color": data["color"], "label": data["label"]}
            for value, data in wetland_categories.items()
            if value != 0  # Exclude "Non Wetland" from legend
        ],
    }

    layer = {
        "id": LAYER_ID,
        "name": "Wetland Types",
        "renderingConfig": rendering_config,
        "paramsConfig": params_config,
        "legendConfig": legend_config,
        "indicator": INDICATOR_ID,
        "type": "INDICATOR",
    }

    return [layer]  # Return as list for consistency


# Create layer data
layers_data = create_wetlands_layer_data()

console.print(f"[green]✅ Created {len(layers_data)} layer record[/green]")
console.print("Layer data:")
console.print_json(data=layers_data[0])

✅ Created 1 layer record

✅ Created 1 layer record

Layer data:

{
  "id": "wetlands-layer",
  "name": "Wetland Types",
  "renderingConfig": {
    "type": "raster",
    "colormap": {
      "type": "categorical",
      "values": [
        0,
        11,
        12,
        14,
        15,
        16,
        21,
        23,
        25,
        26,
        27,
        31,
        35,
        43,
        44,
        45
      ],
      "colors": [
        "#E5CB71",
        "#FF00F3",
        "#000CFF",
        "#CCDE57",
        "#3898DF",
        "#D8255B",
        "#FB00FF",
        "#8A00FF",
        "#FF1E00",
        "#E2FDFF",
        "#9CF7F8",
        "#5DEFAB",
        "#D4E613",
        "#155700",
        "#5B9A68",
        "#408F4B"
      ]
    },
    "opacity": 0.8
  },
  "paramsConfig": {
    "dataSource": "raster",
    "rasterUrl": "/api/tiles/wetlands/{z}/{x}/{y}.png",
    "attribution": "Wetlands classification data"
  },
  "legendConfig": {
    "type": "categorical",
    "title": "Wetland Types",
    "position": "bottom-left",
    "orientation": "vertical",
    "categories": [
      {
        "value": 11,
        "color": "#FF00F3",
        "label": "F1.1 Permanent upland streams"
      },
      {
        "value": 12,
        "color": "#000CFF",
        "label": "F1.2 Permanent lowland rivers"
      },
      {
        "value": 14,
        "color": "#CCDE57",
        "label": "F1.4 Seasonal upland streams"
      },
      {
        "value": 15,
        "color": "#3898DF",
        "label": "F1.5 Seasonal lowland rivers"
      },
      {
        "value": 16,
        "color": "#D8255B",
        "label": "F1.6 Episodic arid rivers"
      },
      {
        "value": 21,
        "color": "#FB00FF",
        "label": "F2.1 Large permanent freshwater lakes"
      },
      {
        "value": 23,
        "color": "#8A00FF",
        "label": "F2.3 Seasonal freshwater lakes"
      },
      {
        "value": 25,
        "color": "#FF1E00",
        "label": "F2.5 Ephemeral freshwater lakes"
      },
      {
        "value": 26,
        "color": "#E2FDFF",
        "label": "F2.6 Permanent salt and soda lakes"
      },
      {
        "value": 27,
        "color": "#9CF7F8",
        "label": "F2.7 Ephemeral salt lakes"
      },
      {
        "value": 31,
        "color": "#5DEFAB",
        "label": "F3.1 Large reservoirs"
      },
      {
        "value": 35,
        "color": "#D4E613",
        "label": "F3.5 Canals ditches and drains"
      },
      {
        "value": 43,
        "color": "#155700",
        "label": "TF1.3 Permanent marshes"
      },
      {
        "value": 44,
        "color": "#5B9A68",
        "label": "TF1.4 Seasonal floodplain marshes"
      },
      {
        "value": 45,
        "color": "#408F4B",
        "label": "TF1.5 Episodic arid floodplains"
      }
    ]
  },
  "indicator": "wetlands-mitigation-potential",
  "type": "INDICATOR"
}

<a id='export-data-to-json-files'></a>
## 4. Export Data to JSON Files

Export all the created data to separate JSON files for CMS import.

In [ ]:
# Export all data to JSON files
console.print(Panel.fit("🚀 Exporting data to JSON files...", style="white"))

# Export each data collection
exports = [
    (locations_data, "locations"),
    (categories_data, "categories"),
    (indicators_data, "indicators"),
    (indicator_data_records, "indicator-data"),
    (layers_data, "layers"),
]

successful_exports = 0
for data, filename in exports:
    try:
        save_json(data, filename)
        successful_exports += 1
    except Exception as e:
        console.print(f"[red]❌ Error exporting {filename}: {e}[/red]")

console.print(
    f"\n[green]✅ Successfully exported {successful_exports}/\
    {len(exports)} files to {CMS_OUTPUT_PATH}[/green]"
)

# List all exported files
console.print("📁 Exported files:")
for file in CMS_OUTPUT_PATH.glob("*.json"):
    size_kb = file.stat().st_size / 1024
    console.print(f"  [cyan]- {file.name}[/cyan] ({size_kb:.1f} KB)")

console.print("\n📄 SQL files:")
for file in CMS_OUTPUT_PATH.glob("*.sql"):
    size_kb = file.stat().st_size / 1024
    console.print(f"  [cyan]- {file.name}[/cyan] ({size_kb:.1f} KB)")

<a id='validation-and-summary'></a>
## 5. Validation and Summary

Validate the generated data and provide a summary of what was created.

In [ ]:
# Validate relationships and data integrity
console.print(Panel.fit("🔍 Validating data integrity...", style="bold yellow"))

validation_results = {
    "locations": len(locations_data),
    "categories": len(categories_data),
    "indicators": len(indicators_data),
    "indicator_data": len(indicator_data_records),
    "layers": len(layers_data),
}

# Check relationships
if indicator_data_records and locations_data:
    # Check if all location references in indicator data exist
    location_ids = {loc["id"] for loc in locations_data}
    indicator_location_ids = {record["location"] for record in indicator_data_records}

    missing_locations = indicator_location_ids - location_ids
    if missing_locations:
        console.print(
            f"[yellow]⚠️  Warning: {len(missing_locations)} location references not found: \
                {missing_locations}[/yellow]"
        )
    else:
        console.print("[green]✅ All location references are valid[/green]")

# Check required fields
required_fields = {
    "categories": ["id", "name"],
    "indicators": ["id", "name", "category"],
    "indicator_data": ["id", "indicator", "location", "data"],
    "layers": ["id", "name", "indicator", "type"],
    "locations": ["id", "name", "code", "type"],
}

all_collections = {
    "categories": categories_data,
    "indicators": indicators_data,
    "indicator_data": indicator_data_records,
    "layers": layers_data,
    "locations": locations_data,
}

for collection_name, required in required_fields.items():
    collection = all_collections[collection_name]
    if collection:
        missing_fields = []
        for record in collection:
            for field in required:
                if field not in record or record[field] is None:
                    missing_fields.append(f"{collection_name}.{field}")

        if missing_fields:
            console.print(f"[red]❌ Missing required fields: {missing_fields}[/red]")
        else:
            console.print(f"[green]✅ {collection_name}: All required fields present[/green]")

# Create summary table
summary_table = Table(title="📊 Data Summary", show_header=True, header_style="bold magenta")
summary_table.add_column("Collection", style="cyan", no_wrap=True)
summary_table.add_column("Records", justify="right", style="green")

for name, count in validation_results.items():
    summary_table.add_row(name.replace("_", " ").title(), str(count))

console.print(summary_table)

╭─────────────────────────────────╮
│ 🔍 Validating data integrity... │
╰─────────────────────────────────╯

✅ All location references are valid

✅ categories: All required fields present

✅ indicators: All required fields present

✅ indicator_data: All required fields present

✅ layers: All required fields present

✅ locations: All required fields present

      📊 Data Summary       
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Collection     ┃ Records ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ Locations      │      22 │
│ Categories     │       1 │
│ Indicators     │       1 │
│ Indicator Data │      22 │
│ Layers         │       1 │
└────────────────┴─────────┘